In [3]:
from sklearn.metrics import  f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import warnings

import mlflow
import os

ModuleNotFoundError: No module named 'matplotlib'

## Constant

In [2]:
DATAPATH = "../../data/data.pkl"
RANDOM_STATE = 42

## Read Data

In [3]:
df = pd.read_pickle(DATAPATH)
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1777056,162,CASH_OUT,183806.32,C691771226,19391.00,0.00,C1416312719,382572.19,566378.51,0,0
1350600,137,PAYMENT,521.37,C203378011,0.00,0.00,M42773300,0.00,0.00,0,0
1991933,179,PAYMENT,3478.18,C1698571270,19853.00,16374.82,M643984524,0.00,0.00,0,0
5092368,355,PAYMENT,1716.05,C913764937,5769.17,4053.13,M1387429131,0.00,0.00,0,0
5066515,354,CASH_IN,253129.93,C2017736577,1328499.49,1581629.42,C407484102,2713220.48,2460090.55,0,0


## EDA

In [ ]:
# What are the different type of transactions ?
df.type.value_counts()

type
CASH_OUT    10840
PAYMENT      6443
TRANSFER     5655
CASH_IN      4227
DEBIT         111
Name: count, dtype: int64

In [8]:
# Target porportion
df.isFraud.value_counts(normalize=True)

isFraud
0    0.698893
1    0.301107
Name: proportion, dtype: float64

In [7]:
# Fraud per type of transactions
df[df.isFraud == 1].type.value_counts(normalize=True)

type
CASH_OUT    0.501157
TRANSFER    0.498843
Name: proportion, dtype: float64

## Data Preprocessing

In [9]:
def wrangle(df):
    df = df.copy()

    cols = []

    # Features leakage
    cols.append("newbalanceDest")
    cols.append("newbalanceOrig")

    # transform step into time
    df["time"] = df["step"].apply(lambda step: (step - 1) % 24 + 1)
    cols.append("step")

    # System Flag
    cols.append("isFlaggedFraud")

    # keep only type of customers M or C
    df["nameOrig"] = df["nameOrig"].str[0]
    df["nameDest"] = df["nameDest"].str[0]
    

    df.drop(columns=cols, inplace=True)
    
    return df

In [10]:
df_preprocessed = wrangle(df)

## Build model

In [11]:
target = "isFraud"
X = df.drop(columns=[target])
y = df[target]

In [12]:
cat_cols = df.select_dtypes("object").columns.tolist()

/var/folders/q0/tlxqcyzj4bl5486b7j7s6ymw0000gn/T/ipykernel_88112/2502784212.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes("object").columns.tolist()


In [13]:
# Split data
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

### Mlflow and Tracking expirement

#### Setup

In [14]:
MLFLOW_URI = ""  # http://127.0.0.1:5000/
EXP_NAME = "fraud_detection"

In [ ]:
# Creation mlruns folder in the current folder 
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXP_NAME)

/Users/serge-nd/IdeaProjects/DSA/.training/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/06/02 22:57:23 INFO mlflow.tracking.fluent: Experiment with name 'fraud_detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/serge-nd/IdeaProjects/DSA/labs/2%20-%20Tracking%20Experiment%20and%20Artifacts%20Management/mlruns/269203051500438586', creation_time=1780433843149, experiment_id='269203051500438586', last_update_time=1780433843149, lifecycle_stage='active', name='fraud_detection', tags={}, workspace='default'>

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="passthrough"  # keeps numeric columns as-is
)

lr_pipe = make_pipeline(
    preprocessor,
    LogisticRegression()
)

#### Tracking Experiment

In [ ]:
with mlflow.start_run():

    #TODO Log dataset (hint: log_input)

    lr_pipe.fit(X_train, y_train)

    #TODO Log metrics (hint: log_metric or log_metrics)


    #TODO Log hyperparameters (hint: log_param or log_params)


    #TODO Log model (hint : sklearn flavor mlflow.log_model )


    #TODO Log confusion matrix figure (hint : log_figure)

#### Model Registry

In [ ]:
#TODO Put the model in the model Registry